# Import libraries

In [1]:
from pyspark.sql.functions import expr
from pyspark.sql.types import IntegerType
from pyspark.sql.functions import upper, col, when

# Step 1 - Read historical data using spark dataframe reader

In [2]:
#Copy the sample code by following the instructions from the manual
customers = spark.read.table("churn_bronze.historical.labelled_customers")


display_cols = ["customerid", "gender", "totalcharges", "churn"]
display(customers.select(display_cols))

# Select the relevant attributes using Code Assist
#### Prompt: Create a dataframe named customers_selected from customers df with only the following columns: customerid, gender, seniorcitizen, partner, dependents, tenure, phoneservice, multiplelines, internetservice, onlinesecurity, onlinebackup, deviceprotection, techsupport, streamingtv, streamingmovies, contract, paperlessbilling, paymentmethod, monthlycharges, totalcharges, churn, year, review

In [3]:
from pyspark.sql.functions import col

# Define the list of columns to select
columns_to_select = [
    "customerid", 
    "gender", 
    "seniorcitizen", 
    "partner", 
    "dependents", 
    "tenure", 
    "phoneservice", 
    "multiplelines", 
    "internetservice", 
    "onlinesecurity", 
    "onlinebackup", 
    "deviceprotection", 
    "techsupport", 
    "streamingtv", 
    "streamingmovies", 
    "contract", 
    "paperlessbilling", 
    "paymentmethod", 
    "monthlycharges", 
    "totalcharges", 
    "churn", 
    "year", 
    "review"
]

# Select the specified columns from the customers DataFrame
customers_selected = customers.select(*columns_to_select)

# Select last 4 years data using Code Assist

### Prompt: Create a dataframe filtered_df from customers_selected with records after 2022

In [4]:
from pyspark.sql.functions import col

# Filter customers_selected to include only records with year > 2022
filtered_df = customers_selected.filter(col("year") > 2022)

# Drop rows with blank columns and NAs

In [5]:
# Import the functions module from pyspark.sql to leverage its built-in functions
from pyspark.sql import functions as F

# Drop rows with missing values from the filtered DataFrame to obtain a cleaned dataset
filtered_customers = filtered_df.dropna()
filtered_customers = filtered_customers.limit(1000)

# Call GenAI model for sentiment analysis of customer reviews

In [6]:
filtered_customers = filtered_customers.withColumn('sentimentScore_str', expr("query_model('default.oci_ai_models.meta.llama-3.2-90b-vision-instruct', concat('What is the sentiment of the review text on a scale of 1 to 5, please give the output as an integer only', review))"))
display(filtered_customers.select('customerid', 'gender', 'monthlycharges', 'totalcharges', 'churn', 'review', 'sentimentScore_str'))

In [7]:
filtered_customers = filtered_customers.withColumn("sentimentScore", filtered_customers["sentimentScore_str"].cast(IntegerType()))

In [8]:
filtered_customers = filtered_customers.withColumn(
    "churn",
    when(upper(col("churn")) == "YES", 1)
    .when(upper(col("churn")) == "NO", 0)
)

# Create schema named historical if one does not exist

In [9]:
spark.sql("CREATE SCHEMA IF NOT EXISTS churn_silver.historical").show()

++
||
++
++



# Write cleansed customer dataframe to silver layer

In [10]:
table_name = "churn_silver.historical.customers"
filtered_customers = filtered_customers.dropna()
filtered_customers.write.mode("overwrite").format("delta").saveAsTable(table_name)

In [11]:
df = spark.read.table(table_name)
df.printSchema()

root
 |-- customerid: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- seniorcitizen: integer (nullable = true)
 |-- partner: string (nullable = true)
 |-- dependents: string (nullable = true)
 |-- tenure: integer (nullable = true)
 |-- phoneservice: string (nullable = true)
 |-- multiplelines: string (nullable = true)
 |-- internetservice: string (nullable = true)
 |-- onlinesecurity: string (nullable = true)
 |-- onlinebackup: string (nullable = true)
 |-- deviceprotection: string (nullable = true)
 |-- techsupport: string (nullable = true)
 |-- streamingtv: string (nullable = true)
 |-- streamingmovies: string (nullable = true)
 |-- contract: string (nullable = true)
 |-- paperlessbilling: string (nullable = true)
 |-- paymentmethod: string (nullable = true)
 |-- monthlycharges: double (nullable = true)
 |-- totalcharges: double (nullable = true)
 |-- churn: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- review: string (nullable = true)
 |-- s